# A checkpoint-gated run, end to endThis runs the whole lifecycle against a simulated instrument: a plan isauthored, the executor runs it, a measurement is acquired over many ticks, therun **holds at a checkpoint**, quality-control rules judge what was acquired,a decision is taken from a closed menu, the plan is amended through the samefile an external agent would use, and the run is released.Nothing here needs an instrument, vendor software or credentials.The simulator is a stand-in, not a model. It emits a layered-sample *shape* sothat a stop condition has something to trigger on and a quality rule hassomething to judge; its numbers are not physical. The paper's claims arechecked against the study's real logged data in `replay/` — see`python -m replay`.

In [ ]:
import sys, tempfile, pathlibsys.path.insert(0, '..')from pace import (Command, ExperimentClient, ExperimentStateFile, Executor,                  MeasurementParams, Peak, QualityCriteria, SafetyEnvelope,                  SimulatedInstrument, StopCondition, decide)run_dir = pathlib.Path(tempfile.mkdtemp(prefix='pace-demo-'))print(run_dir)

## 1. The planA measurement's parameters are a **named set**. Steps refer to the set by namerather than carrying a copy, which is what lets a mid-run correction changeevery step that has not run yet with a single edit.The stop condition is *dynamic*: rather than a fixed scan count, the run watchesa species and stops a set number of scans after it crosses a threshold. A depthprofile has no natural length — the layer thickness is usually the thing beingmeasured — so a fixed count either cuts the profile short or spends beam time inthe substrate.

In [ ]:
film = MeasurementParams(    name='film',    sputter_time_s=2.0,    peaks=[Peak('marker', 18.0), Peak('substrate', 28.0)],    stop_condition=StopCondition(        kind='Dynamic', max_scans=400, label='substrate',        threshold=1000, trigger='rise', post_scans=10),)plan = [    Command('set_temperature', [200.0, None, None]),    Command('measure_now', ['film', 1, 'run_a', 'S1']),    Command('pause', ['QC checkpoint: judge run_a before continuing']),    Command('measure_now', ['film', 2, 'run_b', 'S1']),    Command('shutdown', []),]

## 2. The safety envelopeDeclared before the run and enforced by the engine, below every decision-makinglayer — so a stalled, buggy or absent controller cannot drive the instrumentoutside it.Nothing is defaulted: an undeclared bound is unconstrained, because a guessed"safe" range would either permit something destructive or pin the experiment toa range nobody chose.

In [ ]:
envelope = SafetyEnvelope(temperature=(-150.0, 600.0), stage={'z': (0.0, 20.0)})envelope

### Reject or clamp is decided by provenanceAn **operator's** out-of-range value is rejected, so a mistake in a plansurfaces. An **automatically generated** one is clamped and logged, becauseaborting an unattended run is worse than running at the edge of a declaredrange. Neither silently exceeds.

In [ ]:
try:    envelope.resolve(900.0, 'plan')          # an operator wrote thisexcept Exception as exc:    print('operator  ->', type(exc).__name__, exc)print('automatic ->', envelope.resolve(900.0, 'auto'))

## 3. Run until the checkpoint`Executor.run()` is the whole control flow. A handler returns `IDLE` to becalled again on the next tick and `NEXT` when it is done, so a measurementspanning hundreds of scans never blocks the loop that is also polling for theagent's stop command and writing telemetry.Here the run is `persistent`, so when the queue drains it idle-waits instead ofending. We stop it at the checkpoint by watching the telemetry the executorwrites every tick — which is exactly what an external agent would do.

In [ ]:
store = ExperimentStateFile(run_dir)instrument = SimulatedInstrument(substrate_scan=30, seed=1, log=print)executor = Executor(store, instrument, param_sets={'film': film},                    envelope=envelope, tick_interval=0, log=print)executor.load(plan)# Stop driving the loop once it reaches the checkpoint, so we can inspect it.original = executor.write_statusdef hold_at_the_checkpoint():    original()    if executor._paused:        executor._terminate = Trueexecutor.write_status = hold_at_the_checkpointexecutor.run()print()print('acquired:', instrument.measurements)

## 4. Judge what was acquiredThe quality-control criteria are the rules the agent was instructed to applyduring the study, expressed as code. They judge metrics reduced from themeasurement; `analysis/` is the path from a real depth profile to those numbers,and here we take them from the simulator directly.Every threshold is a constructor argument, because they are properties of onesample, gun and geometry rather than constants.

In [ ]:
criteria = QualityCriteria()for rule, standard, remedy in criteria.describe():    print(f'{rule:18} {standard:58} {remedy}')

In [ ]:
acquired = instrument.measurements[0]metrics = {    'layer_points': 56,                     # from analysis.layer_window on a real profile    'sputter_frames': 11,    'counts_per_px_shot': 0.84,    'uniformity': 0.86,    'stopped_by': acquired.stopped_by,    'source_excursion': False,    'layer_yield': sum(acquired.profile['marker']) / len(acquired.profile['marker']),}findings = criteria.evaluate(metrics)for finding in findings:    print(finding)

## 5. Decide, from a closed menuFive outcomes and no others. That is a design decision rather than adescription: with a closed menu, "what could it have done?" has an answer, andthe run's decisions become a sequence of labels that can be logged, replayed andaudited.

In [ ]:
decision = decide(findings)print(decision)

### What a rejection looks likeHere is the same measurement with too few points across the layer. The decisionchanges to `retune_and_repeat` and carries the corrected parameter — the remedyis the part that changes what happens next.

In [ ]:
undersampled = dict(metrics, layer_points=48)print(decide(criteria.evaluate(undersampled)))

## 6. Amend the plan and release the checkpointThe client is how a decision becomes a change to the run. It writes the same`experiment.json` the executor reads, using a compare-and-swap against a hash ofthe file's bytes — because a read-modify-write here races the executor'swholesale rewrite, and losing that race is silent.

In [ ]:
client = ExperimentClient(run_dir)print('before:', client.get_param_set('film')['sputter_time_s'])client.update_param_set('film', sputter_time_s=1.6)      # the retuneprint('after :', client.get_param_set('film')['sputter_time_s'])client.insert_step(3, 'measure_now', params='film', filename='run_a_repeat')client.send_command('continue_experiment')               # release the holdfor step in client.list_steps():    print(f"{step['index']}  {step['status']:9} {step['action']}")

## 7. Where to go next* `python -m replay` — the real check: the same criteria and menu re-run over  the study's 35 deposited depth profiles, reproducing 35/35 of its  accept-or-reject decisions.* `pace/driver/base.py` — the instrument boundary. Four methods; implement them  and this whole lifecycle runs on other hardware.* `pace/safety.py` — the part that was mechanically enforced during the study,  as distinct from the quality rules above, which were instructions the agent  was given.